In [ ]:
# Рассматриваем как происходит векторный поиск «под капотом». 
# Если кратко, то она учит превращать текст в числа (векторы)
# и находить наиболее близкие по смыслу документы с помощью 
# математики, а не по точному совпадению слов.

# Вместо классического умножения на векторов, мы будем исп умножения
# матрицы на вектор. 

In [1]:
from sentence_transformers import SentenceTransformer
from ingest import load_faq_data

model = SentenceTransformer("all-MiniLM-L6-v2")
documents = load_faq_data()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# 1. Преобразовываем запрос в вектор

question_1 = "Can I still join the course after the start date?"
question_2 = "I just discovered the course, can I still join?"

v1 = model.encode(question_1)
v2 = model.encode(question_2)

In [ ]:
texts = []

for doc in documents:
  text = doc["question"] + " " + doc["answer"]
  texts.append(text)

In [15]:
documents[10]

{'id': '310a39c0f9',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Can I take the course on a locked-down work laptop without admin rights?',
 'answer': 'Yes. Use GitHub Codespaces - you only need a modern browser (or VS Code) and an internet connection, and you can install whatever you need inside the Codespace. GitHub Copilot and OpenAI Codex both work in Codespaces. The free tier is usually generous enough for the course.'}

In [6]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batch = texts[i:i + batch_size]
  batch_vectors = model.encode(batch)
  vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1400

In [31]:
scores = [];

for i in range (len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [32]:
scores[:10]

[np.float32(0.2651543),
 np.float32(0.20766646),
 np.float32(0.08503762),
 np.float32(0.19795904),
 np.float32(0.086557716),
 np.float32(0.21714997),
 np.float32(0.5467235),
 np.float32(0.22926027),
 np.float32(0.2559914),
 np.float32(0.16198394)]

In [16]:
import numpy as np
X = np.array(vectors)

In [ ]:
# 2. Вычисление оценок схожести - тем вычисляется скалярное произведение 
# (dot product) вектора запроса на матрицу X, в которой хранятся векторы
# всех документов (scores = X.dot(v_query)). Каждый элемент в 
# полученном массиве scores — это оценка схожести (косинусное сходство)
# между запросом и конкретным документом.

matrixScore = X.dot(v1)

In [ ]:
# 3. Почему матричное умножение?: В уроке подчеркивается, что X.dot(v_query)
# работает гораздо быстрее, чем цикл по всем документам, потому что 
# numpy использует оптимизированный C-код, а не медленные циклы Python.

In [18]:
matrixScore

array([0.2651543 , 0.20766646, 0.08503762, ..., 0.20289949, 0.13746655,
       0.09721467], shape=(1400,), dtype=float32)

In [ ]:
# 4. Поиск лучших совпадений: После получения оценок нужно выбрать 
# самые релевантные документы. Для поиска одного лучшего совпадения 
# используется np.argmax(scores).

In [19]:
idx = np.argmax(scores)
# argmax выбирает наибольший индетификатор в этой оценке
# наибольший элемент находится 
idx, scores[idx]

(np.int64(510), np.float32(0.762941))

In [20]:
documents[510]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [ ]:
# Как получить 5 лучших результатов - argsort он сортирует идентификаторы 
np.argsort(scores)

# Можно взять последние 5 элементы
top5 = np.argsort(scores)[-5:]
# Мы можем перевернуть функцию (параметры)
top5[::1]

# array([ 515,  905,   77, 1153,  510])

array([ 515,  905,   77, 1153,  510])

In [ ]:
# Чтобы понять список нужно увеличивать его по величине

scores[top5]

# array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
#      dtype=float32)

In [35]:
### Выводим все найденные документы на экран
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.56009996
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date

In [37]:
top5

array([ 515,  905,   77, 1153,  510])

In [ ]:
# Можно даже упростить и сократить код. Например преобразовав 
# минимум в максимум
top5 = np.argsort(-scores)[:5]

In [ ]:
top5